# `omni-asr-lite` 全接口与可插拔多引擎扩展演示 (Demo Notebook)

本 Notebook 演示 **零第三方依赖、可插拔多引擎并发语音转文字与豆包 LLM 无锁分桶流式整理框架 (`omni-asr-lite`)** 的全部调用方式：

1. **环境检查与内置搜狗引擎支持的 14 种语种与方言 (`SOGOU_LANGUAGES`)**
2. **方式一：16 线程极速长音频转写 + 豆包 LLM 无锁分桶流式整理 (`transcribe`)**
3. **方式二：插件扩展 A — 继承 `BaseASREngine` 接入自定义 ASR 引擎并参与多线程协同调度**
4. **方式三：插件扩展 B — 直接注册一个普通 Python 函数 (`register_engine("name", fn)`)**
5. **方式四：模拟某引擎 429 限流时的自适应指数退避熔断与跨引擎自动故障转移**
6. **方式五：同段多引擎赛马择优模式 (`mode="ensemble"`)**
7. **方式六：实时流式转写 (`transcribe_stream`)**
8. **方式七：CLI 命令行调用演示 (`python -m omni_asr_lite`)**

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("src"))

import omni_asr_lite
from omni_asr_lite import (
    ASREngineProtocol,
    BaseASREngine,
    OmniASR,
    SOGOU_LANGUAGES,
    SogouEngine,
    transcribe,
    transcribe_stream,
)

print("omni_asr_lite 版本:", omni_asr_lite.__version__)
print("Python GIL 状态 (False=真多核并行):", getattr(sys, "_is_gil_enabled", lambda: True)())
print("内置搜狗引擎支持的 14 种语种与方言:")
for code, name in SOGOU_LANGUAGES.items():
    print(f"  {code:<22s} -> {name}")

omni_asr_lite 版本: 1.0.0
Python GIL 状态 (False=真多核并行): False
内置搜狗引擎支持的 14 种语种与方言:
  zh-cmn-Hans-CN         -> 普通话
  zh-cmnxyue-Hans-CN     -> 粤语
  zh-yue-Hans-CN         -> 粤语(简体)
  zh-yue-Hant-HK         -> 粤语(繁体/香港)
  zh-cmn-sichuan         -> 四川话
  zh-cmn-hebei           -> 河北话
  zh-cmn-tianjin         -> 天津话
  zh-cmn-dongbei         -> 东北话
  zh-cmn-nanjing         -> 南京话
  zh-cmn-wuhan           -> 武汉话
  zh-cmn-xian            -> 西安话
  zh-cmn-guizhou         -> 贵州话
  zh-cmn-jinan           -> 济南话
  en-US                  -> 英语


---
## 方式一：内置搜狗引擎 16 线程并发 + 豆包 LLM 无锁分桶流式整理 (`transcribe`)

In [2]:
with OmniASR() as asr:
    pcm1 = asr.load_pcm16k("examples/asr_example_zh.wav")
    pcm2 = asr.load_pcm16k("examples/asr_example.wav")
    silence_1s = b"\x00\x00" * 16000
    long_pcm = (pcm1 + silence_1s + pcm2 + silence_1s) * 2

    res = asr.transcribe(
        long_pcm,
        min_segment_sec=4.0,
        max_segment_sec=6.5,
        hotwords=["达摩院"],
        organize=True,
        on_segment=lambda s: print(f"  -> [段 #{s.index:02d} | 引擎={s.engine:<8s} | {s.latency_sec:.2f}s] [{s.timestamp}] {s.text}"),
    )

print(f"\n转写完成！总时长: {res.duration}s | 总耗时: {res.elapsed_sec}s (ASR {res.asr_sec}s + LLM收尾 {res.organize_tail_sec}s)")
print("智能整理成稿:\n" + res.text)

  -> [段 #02 | 引擎=sogou    | 0.25s] [00:05.7 - 00:09.7] 每一天都要快乐哦！
  -> [段 #01 | 引擎=sogou    | 0.30s] [00:00.0 - 00:05.7] 欢迎大家来体验达摩院推出的语音识别模型。
  -> [段 #04 | 引擎=sogou    | 0.31s] [00:15.7 - 00:20.0] 每一天都要快乐哦！
  -> [段 #03 | 引擎=sogou    | 0.45s] [00:09.7 - 00:15.7] 欢迎大家来体验达摩院推出的语音识别模型。

转写完成！总时长: 19.96s | 总耗时: 0.92s (ASR 0.5s + LLM收尾 0.42s)
智能整理成稿:
欢迎大家来体验达摩院推出的语音识别模型。每一天都要快乐哦！

欢迎大家来体验达摩院推出的语音识别模型。每一天都要快乐哦！


---
## 方式二：继承 `BaseASREngine` 接入自定义 ASR 引擎参与多线程协同抢占

In [3]:
class MyCustomBackupEngine(BaseASREngine):
    name = "backup_asr"
    supported_languages = {"zh", "en"}

    def __init__(self, delegate: SogouEngine):
        self._delegate = delegate

    def transcribe_pcm(self, pcm_bytes: bytes, *, language: str = "zh", hotwords=None, context="", timeout=None) -> str:
        return self._delegate.transcribe_pcm(pcm_bytes, language=language, hotwords=hotwords)

with OmniASR() as asr:
    sogou_eng = asr.scheduler.engines["sogou"]
    asr.register_engine(MyCustomBackupEngine(sogou_eng), workers=4, initial_rtf=0.035)
    print("当前已注册引擎列表:", asr.list_engines())

    multi_res = asr.transcribe(
        long_pcm,
        engine_workers={"sogou": 4, "backup_asr": 4},
        min_segment_sec=4.0,
        max_segment_sec=6.5,
        organize=False,
    )
    print("多引擎协同分片统计:", multi_res.engine_stats)

当前已注册引擎列表: ['sogou', 'backup_asr']
多引擎协同分片统计: {'sogou': 3, 'backup_asr': 1}


---
## 方式三：直接将一个普通 Python 函数注册为可调度引擎 (`CallableASREngine`)

In [4]:
with OmniASR() as asr:
    sogou_eng = asr.scheduler.engines["sogou"]
    asr.register_engine("fn_engine", lambda pcm, **kw: sogou_eng.transcribe_pcm(pcm, **kw), workers=2)
    fn_res = asr.transcribe("examples/asr_example_zh.wav", engines=["fn_engine"], split=False, organize=False)
    print("函数式插件引擎识别结果:", fn_res.text, "| 引擎统计:", fn_res.engine_stats)

函数式插件引擎识别结果: 欢迎大家来体验达摩院推出的语音识别模型。 | 引擎统计: {'fn_engine': 1}


---
## 方式四：模拟某引擎 429 限流时的自适应熔断与跨引擎零感知故障转移

In [5]:
with OmniASR() as asr:
    pcm = asr.load_pcm16k("examples/asr_example_zh.wav")
    asr.register_engine(
        "rate_limited_cloud",
        lambda pcm_bytes, **kw: (_ for _ in ()).throw(RuntimeError("HTTP 429 rate limit exceeded")),
        workers=2,
        initial_rtf=0.02,
    )
    fo_res = asr.transcribe(
        pcm + (b"\x00\x00" * 16000) + pcm,
        engines=["rate_limited_cloud", "sogou"],
        engine_workers={"rate_limited_cloud": 2, "sogou": 2},
        min_segment_sec=4.0,
        max_segment_sec=6.5,
        organize=False,
    )
    for s in fo_res.segments:
        print(f"段 #{s.index:02d}: 最终引擎={s.engine} | 容灾链路={s.failover_chain} | 文本={s.text}")
    print("遥测统计:", fo_res.engine_metrics)

段 #01: 最终引擎=sogou(failover) | 容灾链路=['rate_limited_cloud', 'sogou', 'sogou(failover)'] | 文本=欢迎大家来体验达摩院推出的语音识别模型。
段 #02: 最终引擎=sogou | 容灾链路=['sogou'] | 文本=欢迎大家来体验达摩院推出的语音识别模型。
遥测统计: {'rate_limited_cloud': {'name': 'rate_limited_cloud', 'workers': 2, 'succeeded': 0, 'failed': 1, 'retried_away': 0, 'hedged_wins': 0, 'rate_limited': 1, 'circuit_trips': 1, 'total_audio_sec': 0.0, 'total_latency_sec': 0.0, 'ewma_rtf': 0.04, 'avg_speed_x': 0.0}, 'sogou': {'name': 'sogou', 'workers': 2, 'succeeded': 2, 'failed': 0, 'retried_away': 0, 'hedged_wins': 0, 'rate_limited': 0, 'circuit_trips': 0, 'total_audio_sec': 12.09, 'total_latency_sec': 0.6545918001793325, 'ewma_rtf': 0.043, 'avg_speed_x': 18.5}}


---
## 方式五：同段多引擎赛马择优模式 (`mode="ensemble"`)

In [6]:
with OmniASR() as asr:
    sogou_eng = asr.scheduler.engines["sogou"]
    asr.register_engine(MyCustomBackupEngine(sogou_eng), workers=2)
    ens_res = asr.transcribe("examples/asr_example_zh.wav", mode="ensemble", split=False, organize=True)
    seg0 = ens_res.segments[0]
    print("胜出引擎:", seg0.engine)
    print("同段候选对比:", seg0.candidates)
    print("最终文本:", ens_res.text)

胜出引擎: ensemble(sogou)
同段候选对比: {'sogou': '欢迎大家来体验达摩院推出的语音识别模型。', 'backup_asr': '欢迎大家来体验达摩院推出的语音识别模型。'}
最终文本: 欢迎大家来体验达摩院推出的语音识别模型。


---
## 方式六：实时流式转写 (`transcribe_stream`)

In [7]:
for ev in transcribe_stream("examples/asr_example.wav", chunk_sec=1.0):
    print(f"[{ev.type.upper():7s} | {ev.engine}] [{ev.timestamp}] {ev.text} (delta={ev.delta!r})")

[PARTIAL | sogou] [00:00.0 - 00:01.0] 每一天 (delta='每一天')
[PARTIAL | sogou] [00:00.0 - 00:02.0] 每一天都要快乐哦！ (delta='都要快乐哦！')
[FINAL   | sogou] [00:00.0 - 00:02.4] 每一天都要快乐哦！ (delta='')


---
## 方式七：CLI 命令行调用演示 (`python -m omni_asr_lite`)

In [8]:
import subprocess

cmd = [sys.executable, "-m", "omni_asr_lite", "examples/asr_example.wav", "-f", "srt"]
env = dict(os.environ, PYTHONPATH=os.path.abspath("src"), PYTHONIOENCODING="utf-8")
proc = subprocess.run(cmd, stdin=subprocess.DEVNULL, capture_output=True, text=True, encoding="utf-8", env=env)
print(proc.stdout)

[01 | sogou   ] [00:00.0 - 00:02.4] 每一天都要快乐哦！
1
00:00:00,000 --> 00:00:02,430
每一天都要快乐哦！


